In [1]:
import os
import glob
import pandas as pd
import re
from whoosh import index
from whoosh.fields import Schema, TEXT, ID
from whoosh.qparser import QueryParser, OrGroup
from whoosh import query as Q
import shutil

# -----------------------------
# 1. Set working directory
# -----------------------------
os.chdir("/Users/simon/Documents/repo/cities-learning-dec")

# -----------------------------
# 2. Read first OA CSV for testing
# -----------------------------
file_pattern = "/Users/simon/Documents/repo/cities-learning/data/OpenAlex/05_deduplicated/city_works_df_NA_abstr_added_dedup_*.csv"
file_names = glob.glob(file_pattern)

all_oas = [pd.read_csv(f) for f in file_names]
oa = pd.concat(all_oas, ignore_index=True)

# Make sure abstract column exists
assert 'abstract' in oa.columns, "Column 'abstract' not found"

# -----------------------------
# 3. Read climate solution typology
# -----------------------------
typology_file = "./data/climate_solutions_typology/climate_solution_typology_long.xlsx"
typology = pd.read_excel(typology_file)
typology['solution_name'] = typology['solution_name'].str.strip()
assert 'search_string' in typology.columns, "Column 'search_string' not found"

# -----------------------------
# 4. Split abstracts into sentences
# -----------------------------
import spacy

# Load the pretrained English model (small model is enough for sentence splitting)
nlp = spacy.load("en_core_web_sm")

sentences = []
sentence_id = []
id = []

for i, abstract in oa['abstract'].items():
    if pd.isna(abstract):
        continue
    
    doc = nlp(str(abstract))
    for j, sent in enumerate(doc.sents, start=1):
        oa_id = oa.at[i, 'id']
        id.append(oa_id)  
        sentences.append(sent.text.strip())
        sentence_id.append(f"{oa_id}_s{j}")
        

# Create a DataFrame for sentences
sent_df = pd.DataFrame({
    'sentence_id': sentence_id,
    'id': id,
    'sentence': sentences
})


# -----------------------------
# 5. Create Whoosh index for sentences
# -----------------------------
schema = Schema(id=ID(stored=True), sentence=TEXT(stored=True))
if not os.path.exists("data/climate_solutions_typology/indexdir_sent"):
    os.mkdir("data/climate_solutions_typology/indexdir_sent")
ix_sent = index.create_in("data/climate_solutions_typology/indexdir_sent", schema)

writer = ix_sent.writer()
for i, row in sent_df.iterrows():
    writer.add_document(id=str(row['sentence_id']), sentence=row['sentence'])
writer.commit()

# -----------------------------
# 6. Search sentences for each climate solution
# -----------------------------
results_sent = {}

for idx, row in typology.iterrows():
    solution_name = row['solution_name']
    solution_id = row['solution_id']
    search_string = row['search_string']

    print(f"Searching for solution: {solution_name}")

    with ix_sent.searcher() as searcher:
        # exact phrases using Phrase class (slop=0)
        qp = QueryParser("sentence", schema=ix_sent.schema, group=OrGroup.factory(0), phraseclass=Q.Phrase)
        q = qp.parse(search_string)
        hits = searcher.search(q, limit=None)
        match_ids = set(hit['id'] for hit in hits)

        # Create a boolean column for matches
        sent_df[f'solution_{solution_id}_match'] = sent_df['sentence_id'].isin(match_ids)

# -----------------------------
# 7. Inspect results
# -----------------------------
print(sent_df.head(20))

# -----------------------------
# 7. Check results in more detail for a few solutions
# -----------------------------
def print_matching_abstracts_from_typology(oa_df, typology_df, solution_id, column="abstract", n=5):
    # solution_name_clean = solution_name.strip()  # remove trailing spaces from input
    match_col = f"solution_{solution_id}_match"
    print(f"Looking for match column: {match_col}")
    solution_name = typology_df[typology_df["solution_id"] == solution_id]["solution_name"]
    print(solution_name)

    # Strip spaces in typology when matching
    search_row = typology_df.loc[typology_df['solution_id'] == solution_id]

    if search_row.empty:
        print(f"No search string found for solution '{solution_id}'")
        return
    search_string = search_row['search_string'].values[0]

    if match_col not in oa_df.columns:
        print(f"No match column found for solution '{solution_id}'")
        return

    matched_abstracts = oa_df.loc[oa_df[match_col], column]

    if matched_abstracts.empty:
        print(f"No abstracts matched for solution '{solution_id}'")
        return

    print(f"\n--- Sample abstracts matching '{solution_name}' ---")
    print(f"Search query: {search_string}\n")

    for i, abstract in enumerate(matched_abstracts.head(n), 1):
        print(f"{i}. {abstract}\n")

print_matching_abstracts_from_typology(sent_df, typology, solution_id=1, column="sentence", n=4)

# -----------------------------
# 8. store the result
# -----------------------------
# Select all columns ending with '_match' plus the 'id' column
match_cols = [col for col in sent_df.columns if col.endswith('_match')]
sent_df[match_cols] = sent_df[match_cols]*1
cols_to_save = ['id', 'sentence_id'] + match_cols

# Save to CSV
sent_df[cols_to_save].to_parquet("data/climate_solutions_typology/oa_sentence_solutions.parquet", index=False)

print(f"Saved {len(match_cols)} solution match columns + 'id' to oa_solutions.csv")




Searching for solution: Switch to cycling and walking
Searching for solution: Highly accessible compact urban form and transit networks, and associated (AI-based) urban planning strategies
Searching for solution: Uptake of BEVs and electric 2 and 3-wheelers
Searching for solution: Switch to public transit
Searching for solution: Low-carbon infrastructure materials
Searching for solution: Shared mobility
Searching for solution: Teleworking
Searching for solution: Energy-efficient lifestyle
Searching for solution: Compact urban form
Searching for solution: High efficiency appliances
Searching for solution: Limiting growth in floor space/sufficiency
Searching for solution: Low-carbon building materials
Searching for solution: Smart home systems (incl. smart thermostats and HVAC)
Searching for solution: High-efficiency building envelopes and passive houses
Searching for solution: Heat pumps and heat recovery systems
Searching for solution: nZEBs
Searching for solution: Citizen and renewabl

In [2]:
# -----------------------------
# 9. store sentence-level data
# -----------------------------
cols_to_save = ['sentence_id', 'id', 'sentence']

sent_df[cols_to_save].to_parquet(
    "data/climate_solutions_typology/oa_sentences.parquet",
    index=False
)

print(f"Saved {len(sent_df)} sentences with 'sentence_id', 'id', and 'sentence' to oa")

Saved 3175322 sentences with 'sentence_id', 'id', and 'sentence' to oa


In [1]:
# -----------------------------
# 10. delete indexed sentences that are memory-heavy and no longer needed
# -----------------------------

index_dir = "data/climate_solutions_typology/indexdir_sent"

if os.path.exists(index_dir):
    shutil.rmtree(index_dir)  # deletes folder and all contents
    print(f"Deleted index directory: {index_dir}")

Deleted index directory: data/climate_solutions_typology/indexdir_sent
